<a href="https://colab.research.google.com/github/NissanEx/analisa-kinerja-prabowo/blob/main/AnalisaKepuasanMasyarakt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install apify-client pandas transformers torch

In [ ]:
import os
from apify_client import ApifyClient

# MASUKKAN TOKEN DI SINI:
APIFY_TOKEN = "TARUH TOKENNYA SINI BUNG"

os.environ["APIFY_TOKEN"] = APIFY_TOKEN
client = ApifyClient(APIFY_TOKEN)
print("Klien Apify berhasil diinisialisasi.")

In [ ]:
# Cell 3: Scraping Data (Multi-Platform) + Tagging Topik Pemerintahan + Deteksi Meme
import pandas as pd
import re
import datetime
import numpy as np

# --- PENGAMAN APIFY ---
# Cek apakah 'client' sudah didefinisikan dari Cell sebelumnya
try:
    client
except NameError:
    print("⚠️ PERINGATAN: Variabel 'client' (Apify) belum dibuat.")
    print("   Sistem akan menggunakan data dummy untuk preview.")
    client = None

# ---------- KONFIGURASI ----------
MAX_PER_PLATFORM = 100
MIN_ASLI = 5

# PERUBAHAN 1: Keyword spesifik program & isu pemerintahan Indonesia
KEYWORDS = [
    # Program & kebijakan konkret
    "program pemerintah", "bansos PKH", "bantuan sosial", "BPJS Kesehatan",
    "KIP Kuliah", "Kartu Prakerja", "subsidi BBM", "subsidi listrik",
    "IKN Nusantara", "makan bergizi gratis", "dana desa", "bantuan UMKM",
    "tol trans jawa", "kereta cepat whoosh", "sertifikat tanah", "vaksinasi",
    "reformasi birokrasi", "pemberantasan korupsi",
    # Kritik & keluhan rakyat
    "kinerja pemerintah", "kritik pemerintah", "keluhan rakyat", "pelayanan publik",
    "harga sembako naik", "gaji UMR", "lapangan kerja", "jalan rusak",
    "banjir", "korupsi pejabat", "janji politik", "pajak naik", "PHK buruh",
    # Meme & sindiran
    "meme politik", "sindiran pemerintah", "pemerintah gagal",
    "katanya pemerintah", "janji manis", "omong kosong",
]

PLATFORMS = {
    "Facebook": {
        "actor": "apify/facebook-posts-scraper",
        "input": {
            "startUrls": [
                {"url": "https://www.facebook.com/detikcom"},
                {"url": "https://www.facebook.com/kompascom"},
                {"url": "https://www.facebook.com/cnnindonesia"},
            ],
            "resultsLimit": MAX_PER_PLATFORM,
        },
        "text_keys": ["text", "message"],
    },
    "Instagram": {
        "actor": "apify/instagram-hashtag-scraper",
        "input": {"hashtags": [k.replace(" ", "") for k in KEYWORDS],
                  "resultsLimit": MAX_PER_PLATFORM},
        "text_keys": ["caption", "text"],
    },
    "TikTok": {
        "actor": "clockworks/tiktok-scraper",
        "input": {"searchQueries": KEYWORDS,
                  "resultsPerPage": max(10, MAX_PER_PLATFORM // len(KEYWORDS))},
        "text_keys": ["text", "desc"],
    },
    "Twitter/X": {
        "actor": "apidojo/tweet-scraper",
        "input": {"searchTerms": KEYWORDS, "maxItems": MAX_PER_PLATFORM, "sort": "Latest"},
        "text_keys": ["fullText", "text"],
    },
}

# ---------- PERUBAHAN 2: Data dummy diperluas, spesifik program pemerintahan ----------
data_dummy = [
    # === BANSOS & PERLINSOS ===
    {"platform": "Facebook", "text": "Bansos PKH tahap 3 kapan cair? Sudah 2 bulan nunggak, anak mau sekolah butuh biaya. Tolong respon pak."},
    {"platform": "Twitter/X", "text": "Katanya bansos tepat sasaran, tapi tetangga saya yang punya motor baru malah dapat. Yang miskin beneran nggak kebagian. Klasik."},
    {"platform": "Instagram", "text": "Program BPNT sangat membantu warung saya, sekarang bisa restock sembako. Terima kasih pemerintah!"},
    {"platform": "TikTok", "text": "POV: Kamu rakyat kecil, bansos katanya cair tapi harus bayar administrasi dulu. (Meme)"},
    {"platform": "Twitter/X", "text": "Bansos ada tapi antri berjam-jam, syaratnya ribet, dan harus lewat calo. Ini namanya membantu atau menyiksa?"},
    # === KESEHATAN (BPJS/KIS) ===
    {"platform": "Facebook", "text": "Antri BPJS di rumah sakit 4 jam, sakit malah tambah parah. Tolong benahi layanan kesehatan publik!"},
    {"platform": "Instagram", "text": "Vaksinasi gratis berjalan lancar! Pelayanannya ramah dan cepat. Mantap!"},
    {"platform": "TikTok", "text": "Dokter bilang harus operasi, tapi antrian BPJS baru 3 bulan lagi. Ngenes bener. (Meme)"},
    {"platform": "Twitter/X", "text": "KIS sudah bagus, tapi rumah sakit swasta banyak yang nolak pasien BPJS. Percuma punya kartu kalau nggak diterima."},
    {"platform": "Facebook", "text": "Program KIS sangat membantu keluarga saya. Anak saya bisa berobat gratis. Terima kasih pemerintah!"},
    # === PENDIDIKAN (KIP/BEASISWA) ===
    {"platform": "Facebook", "text": "Pendaftaran sekolah negeri ribet, ujung-ujungnya bayar uang gedung. Padahal katanya gratis."},
    {"platform": "Twitter/X", "text": "Program beasiswa KIP Kuliah sangat membantu adik saya. Semoga lancar terus, terima kasih pemerintah!"},
    {"platform": "TikTok", "text": "KIP Kuliah cair, tapi harus upload 100 dokumen. Adminnya slow respon. (Sindiran)"},
    {"platform": "Instagram", "text": "Sekolah gratis katanya, tapi buku, seragam, dan kegiatan tetap bayar. Pungli dibalut program. 🙄"},
    # === EKONOMI, HARGA & SUBSIDI ===
    {"platform": "Facebook", "text": "Harga sembako naik terus, gaji UMR segini mana bisa makan enak. Kapan rakyat kecil sejahtera?"},
    {"platform": "Instagram", "text": "Gaji UMR Jakarta naik tapi harga ayam geprek udah 15rb. Hidup di Ibu Kota tuh bonus drama ya. 😭"},
    {"platform": "Twitter/X", "text": "Subsidi BBM katanya naik, tapi harga di SPBU tetap aja. Uang rakyat ke mana pak?"},
    {"platform": "TikTok", "text": "Ekonomi katanya stabil, tapi dompet gue: 📉📉📉 (Meme)"},
    {"platform": "Twitter/X", "text": "Pajak naik terus, tapi jalanan depan komplek rusak parah. Uang rakyat pada ke mana coba?"},
    {"platform": "Facebook", "text": "Program bantuan UMKM sangat membantu, usaha saya sekarang bisa buka cabang baru. Terima kasih!"},
    {"platform": "Twitter/X", "text": "Pemerintah: 'Kita sudah berhasil menekan inflasi'. Aku: *liat harga cabe 100rb/kg*. Yaudah lah ya."},
    # === INFRASTRUKTUR (TOL, KERETA CEPAT, IKN, JALAN) ===
    {"platform": "Facebook", "text": "Jalan tol baru bagus banget, tapi jalan desa di kampung saya hancur penuh lubang."},
    {"platform": "Instagram", "text": "Banjir lagi, banjir lagi. Pemerintah cuma nyalahin hujan, padahal drainase dibiarkan mampet. 🙄"},
    {"platform": "Twitter/X", "text": "Proyek kereta cepat Whoosh bagus, tapi tolong lirik transportasi umum di daerah yang masih angkot tua."},
    {"platform": "TikTok", "text": "POV: Kamu jalan-jalan ke luar negeri, jalanannya mulus. Pas balik ke Indonesia, gigi goyang. (Meme)"},
    {"platform": "Twitter/X", "text": "IKN Nusantara dibangun triliunan, tapi desa tetangga belum ada listrik. Prioritas apa ini pak?"},
    {"platform": "Instagram", "text": "Tol Trans Jawa memudahkan mudik, terima kasih pemerintah! Lanjutkan pembangunan."},
    # === HUKUM & KORUPSI ===
    {"platform": "Twitter/X", "text": "Hukum tajam ke bawah tumpul ke atas. Rakyat kecil salah dikit dipenjara, pejabat korupsi bebas."},
    {"platform": "Facebook", "text": "Korupsi merajalela, rakyat makin susah. Kapan negara ini bebas dari para tikus berdasi?"},
    {"platform": "Twitter/X", "text": "Sarkas dikit: 'Kami mendengar aspirasi rakyat'. Padahal setelah dipilih, rakyat dilupakan. Klasik."},
    {"platform": "TikTok", "text": "Meme bapak tani tersenyum padahal gagal panen. Terima kasih pak, kami sangat bahagia. 🥲"},
    # === LAYANAN PUBLIK & ADMINISTRASI ===
    {"platform": "Facebook", "text": "Bikin KTP sekarang cuma 10 menit, keren banget Dukcapil!"},
    {"platform": "Instagram", "text": "Kenapa antrian di kantor pajak masih lama? Sistemnya sering error, sangat mengecewakan. 😡"},
    {"platform": "Twitter/X", "text": "Info bansos tahap 2 kapan cair? Tolong respon cepat dong pemerintah. Udah nunggak bayar kontrakan."},
    {"platform": "TikTok", "text": "Aplikasi layanan pemerintah ini mudah digunakan dan responsif. Sangat memudahkan urusan saya."},
    {"platform": "Instagram", "text": "Layanan perizinan sekarang bisa online, hemat waktu. Awalnya ragu, ternyata cepat!"},
    # === KETENAGAKERJAAN ===
    {"platform": "Twitter/X", "text": "Susah banget cari kerja sekarang, lapangan kerja makin sempit. Yang ada cuma janji manis saat kampanye."},
    {"platform": "Facebook", "text": "Terima kasih program pelatihan kerja, saya sekarang punya keterampilan dan dapat pekerjaan tetap."},
    {"platform": "TikTok", "text": "PHK di mana-mana, tapi menteri bilang ekonomi tumbuh. Oh iya ternyata saya yang salah perasaan, maaf pak. (Meme)"},
    # === POLITIK & JANJI ===
    {"platform": "Instagram", "text": "Politik itu kayak seblak: pedas, bikin sakit perut, tapi banyak yang doyan. (Sindiran)"},
    {"platform": "Twitter/X", "text": "Dibilang demo anarkis, padahal cuma mau nanya hak rakyat yang dirampas. Oke sip."},
    {"platform": "TikTok", "text": "POV: Kamu pejabat, ditegur rakyat malah ngeles. (Meme)"},
    {"platform": "Facebook", "text": "Kinerja pemerintah saat ini sudah cukup baik dalam pembangunan infrastruktur digital."},
    {"platform": "Instagram", "text": "Suka sedih liat pengemis, tapi pemerintah malah sibuk beli jet tempur. Prioritas apa ini?"},
    {"platform": "Facebook", "text": "Program bantuan air bersih di desa kami sangat bermanfaat. Anak-anak sekarang nggak sakit perut lagi."},
    {"platform": "Twitter/X", "text": "Tolong dong pak, harga tiket pesawat mahal banget. Mau mudik aja mikir 10 kali."},
    {"platform": "TikTok", "text": "Pemerintah minta kita bersabar, tapi sabar ada batasnya pak. Perut nggak bisa diajak sabar. 🤬"},
]

# ---------- PERUBAHAN 3: Kamus topik program pemerintahan ----------
TOPIK_KEYWORDS = {
    "Bansos & Perlinsos": ["bansos", "pkh", "bantuan sosial", "blt", "bpnt", "sembako", "bantuan", "perlinsos"],
    "Kesehatan (BPJS/KIS)": ["bpjs", "kis", "rumah sakit", "vaksin", "kesehatan", "puskesmas", "dokter", "berobat"],
    "Pendidikan (KIP/Beasiswa)": ["kip", "beasiswa", "sekolah", "kuliah", "pendidikan", "guru", "siswa", "uang gedung"],
    "Ekonomi & Harga": ["harga", "sembako", "inflasi", "subsidi", "bbm", "umr", "gaji", "pajak", "ekonomi", "umkm", "cabe"],
    "Infrastruktur": ["jalan", "tol", "jembatan", "kereta", "whoosh", "ikn", "pembangunan", "infrastruktur", "drainase", "banjir"],
    "Korupsi & Hukum": ["korupsi", "kpk", "hukum", "penjara", "tikus", "suap", "kasus", "demo", "anarkis"],
    "Layanan Publik": ["ktp", "dukcapil", "antri", "pelayanan", "perizinan", "online", "administrasi", "pajak"],
    "Ketenagakerjaan": ["kerja", "lapangan kerja", "phk", "buruh", "upah", "prakerja", "pelatihan"],
    "Politik & Janji": ["janji", "kampanye", "politik", "dipilih", "menteri", "presiden", "dpr", "demo", "aspirasi"],
}

# ---------- PERUBAHAN 4: Pola meme/sindiran ----------
MEME_PATTERNS = [
    r"\bpov\b", r"\bmeme\b", r"\bceritanya\b", r"\(meme\)", r"\(sindiran\)",
    r"\(sarkas\)", r"lucu tapi nyata", r"padahal", r"oh iya", r"maaf pak",
    r"yaudah lah", r"terima kasih pak.*(gagal|rusak|kosong|susah|mahal)",
    r"kami (sangat )?bahagia", r"klasik", r"oke sip", r"katanya",
]

# ---------- FUNGSI BANTU ----------
COLS = ["platform", "text", "sumber", "author", "created_at",
        "followers", "acc_created", "url"]
RAW = {}

def dapat(item, *paths):
    for p in paths:
        v = item
        for k in p.split("."):
            v = v.get(k) if isinstance(v, dict) else None
            if v is None:
                break
        if v is not None and v != "":
            return v
    return None

def ambil_teks(item, keys):
    for k in keys:
        v = item.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None

def get_dataset_id(run):
    if isinstance(run, dict):
        return run.get("defaultDatasetId") or run.get("default_dataset_id")
    return getattr(run, "default_dataset_id", None) or getattr(run, "defaultDatasetId", None)

def jalankan_actor(actor, run_input):
    if client is None:
        raise RuntimeError("Client Apify tidak tersedia.")
    try:
        return client.actor(actor).call(run_input=run_input, logger=None)
    except TypeError:
        return client.actor(actor).call(run_input=run_input)

def deteksi_topik(teks):
    t = teks.lower()
    for topik, kws in TOPIK_KEYWORDS.items():
        if any(kw in t for kw in kws):
            return topik
    return "Lainnya"

def deteksi_meme(teks):
    return bool(re.search("|".join(MEME_PATTERNS), teks, flags=re.I))

def scrape_platform(nama, cfg):
    run = jalankan_actor(cfg["actor"], cfg["input"])
    if run is None:
        raise RuntimeError("actor tidak mengembalikan run")
    ds_id = get_dataset_id(run)
    if not ds_id:
        raise RuntimeError("dataset id tidak ditemukan")
    items = list(client.dataset(ds_id).iterate_items())
    RAW[nama] = items
    print(f"   📦 {len(items)} item mentah")

    rows = []
    for it in items:
        try:
            t = ambil_teks(it, cfg["text_keys"])
            if not t:
                continue
            rows.append({
                "platform": nama, "text": t, "sumber": "asli",
                "author": dapat(it, "authorMeta.name", "author.userName", "author.name",
                                "ownerUsername", "user.name", "user.screen_name", "pageName"),
                "created_at": dapat(it, "createTimeISO", "createdAt", "timestamp", "time"),
                "followers": dapat(it, "authorMeta.fans", "author.followers",
                                   "followersCount", "user.followers_count"),
                "acc_created": dapat(it, "author.createdAt", "user.created_at"),
                "url": dapat(it, "webVideoUrl", "url", "postUrl", "permalink"),
            })
        except Exception:
            continue
    return pd.DataFrame(rows, columns=COLS)

# ---------- JALANKAN SCRAPING ----------
df_dummy = pd.DataFrame(data_dummy)
df_dummy["sumber"] = "dummy"
hasil = []

for nama, cfg in PLATFORMS.items():
    print(f"⏳ Scraping {nama}...")
    try:
        df_p = scrape_platform(nama, cfg)
        print(f"   ✅ {len(df_p)} data asli dengan teks")
    except Exception as e:
        print(f"   ⚠️ Gagal: {type(e).__name__}: {e}")
        df_p = pd.DataFrame(columns=COLS)

    if len(df_p) < MIN_ASLI:
        print(f"   🔄 {nama}: data asli < {MIN_ASLI}, tambah dummy {nama}")
        df_p = pd.concat([df_p, df_dummy[df_dummy["platform"] == nama]], ignore_index=True)
    hasil.append(df_p)

df_medsos = pd.concat([h for h in hasil if not h.empty], ignore_index=True)

# ---------- BERSIHKAN TEKS ----------
n0 = len(df_medsos)
df_medsos["text"] = df_medsos["text"].astype(str).str.strip()
df_medsos = df_medsos[df_medsos["text"].str.len() >= 15]
df_medsos = df_medsos.drop_duplicates(subset="text").reset_index(drop=True)
print(f"\n🧹 Cleaning: {n0} → {len(df_medsos)} data")

if df_medsos.empty:
    df_medsos = df_dummy.copy()

# ---------- PERUBAHAN 7: TAG TOPIK & MEME ----------
df_medsos["topik_program"] = df_medsos["text"].apply(deteksi_topik)
df_medsos["meme_flag"] = df_medsos["text"].apply(deteksi_meme)


# =======================================================
# REVISI CLEANING METADATA (ANTI NONE & TIMESTAMP)
# =======================================================
def format_waktu(val):
    if pd.isna(val) or val is None or val == "-":
        return "-"
    try:
        # Cek jika val berupa angka (unix timestamp)
        val_float = float(val)
        if val_float > 1e11: # Jika format milidetik, bagi 1000
            val_float = val_float / 1000
        return datetime.datetime.fromtimestamp(val_float).strftime("%Y-%m-%d %H:%M")
    except (ValueError, TypeError):
        # Jika bukan angka, kembalikan sebagai teks (mungkin sudah format tanggal)
        return str(val)

# 1. Format Tanggal
if "created_at" in df_medsos.columns:
    df_medsos["created_at"] = df_medsos["created_at"].apply(format_waktu)

# 2. Bersihkan Author (Ubah None jadi "Anonim")
df_medsos["author"] = df_medsos["author"].fillna("Anonim").astype(str)
df_medsos["author"] = df_medsos["author"].replace(["None", "nan", "NaN"], "Anonim")

# 3. Bersihkan Followers (Ubah None jadi 0 dan paksa jadi angka)
df_medsos["followers"] = df_medsos["followers"].fillna(0)
df_medsos["followers"] = pd.to_numeric(df_medsos["followers"], errors="coerce").fillna(0).astype(int)

# 4. Bersihkan Acc Created & URL (Ubah None jadi "-")
df_medsos["acc_created"] = df_medsos["acc_created"].fillna("-").astype(str)
df_medsos["acc_created"] = df_medsos["acc_created"].replace(["None", "nan", "NaN"], "-")

df_medsos["url"] = df_medsos["url"].fillna("-").astype(str)
df_medsos["url"] = df_medsos["url"].replace(["None", "nan", "NaN"], "-")
# =======================================================


# ---------- SIMPAN & TAMPILKAN ----------
df_medsos.to_csv("data_medsos.csv", index=False)
print("💾 Tersimpan: data_medsos.csv")

print(f"\nTotal data siap dianalisis: {len(df_medsos)}")
display(df_medsos.groupby(["platform", "sumber"]).size().unstack(fill_value=0))

print("\nDistribusi topik program pemerintahan:")
display(df_medsos["topik_program"].value_counts())
print(f"\nKonten meme/sindiran: {int(df_medsos['meme_flag'].sum())} dari {len(df_medsos)}")

print("\nPreview Data (Cek kolom created_at, followers, dll):")
display(df_medsos.head(10))

In [ ]:
# Cell 4: Analisis Sentimen (Ensemble) + Koreksi Sarkasme + Deteksi Kritik
import re
import torch
import numpy as np
import pandas as pd
from transformers import pipeline

# ---------- KONFIGURASI ----------
USE_ENSEMBLE = True
LOW_CONF_THRESHOLD = 0.55
BATCH_SIZE = 32
MAX_LEN = 128

possible_text_cols = ['text', 'message', 'postText', 'caption', 'content']
text_col = next((c for c in possible_text_cols if c in df_medsos.columns), None)

if not text_col:
    print(f"❌ Kolom teks tidak ditemukan. Kolom yang ada: {df_medsos.columns.tolist()}")
else:
    print(f"✅ Menggunakan kolom '{text_col}'")

    # ---------- PREPROCESSING ----------
    def bersihkan(t):
        t = str(t)
        t = re.sub(r"http\S+|www\.\S+", " ", t)
        t = re.sub(r"@\w+", " ", t)
        t = re.sub(r"#(\w+)", r"\1", t)
        t = re.sub(r"\((meme|sindiran|sarkas)\)", " ", t, flags=re.I)
        t = re.sub(r"(.)\1{3,}", r"\1\1", t)
        t = re.sub(r"\s+", " ", t).strip()
        return t

    df_medsos = df_medsos.dropna(subset=[text_col]).reset_index(drop=True)
    df_medsos["text_clean"] = df_medsos[text_col].astype(str).apply(bersihkan)
    df_medsos = df_medsos[df_medsos["text_clean"].str.len() >= 5].reset_index(drop=True)
    texts = df_medsos["text_clean"].tolist()

    # ---------- LOAD MODEL ----------
    device = 0 if torch.cuda.is_available() else -1
    print(f"⏳ Memuat model... (device: {'GPU' if device == 0 else 'CPU'})")

    MODELS = [("w11wo/indonesian-roberta-base-sentiment-classifier", None)]
    if USE_ENSEMBLE:
        MODELS.append(("cardiffnlp/twitter-xlm-roberta-base-sentiment",
                       {"LABEL_0": "negatif", "LABEL_1": "netral", "LABEL_2": "positif"}))

    LABELS = ["negatif", "netral", "positif"]

    def norm_label(label, label_map):
        if label_map and label in label_map:
            return label_map[label]
        l = label.lower()
        if "pos" in l: return "positif"
        if "neg" in l: return "negatif"
        return "netral"

    def prediksi_probs(model_name, label_map):
        pipe = pipeline("sentiment-analysis", model=model_name, device=device)
        outs = pipe(texts, top_k=None, truncation=True,
                    max_length=MAX_LEN, batch_size=BATCH_SIZE)
        probs = np.zeros((len(texts), 3))
        for i, out in enumerate(outs):
            for d in out:
                probs[i, LABELS.index(norm_label(d["label"], label_map))] = d["score"]
        return probs

    all_probs = []
    for name, lmap in MODELS:
        print(f"⏳ Menganalisis dengan {name} ...")
        all_probs.append(prediksi_probs(name, lmap))

    probs = np.mean(all_probs, axis=0)
    idx = probs.argmax(axis=1)
    df_medsos["sentiment"] = [LABELS[i].capitalize() for i in idx]
    df_medsos["sentiment_score"] = probs.max(axis=1).round(4)
    df_medsos["low_confidence"] = df_medsos["sentiment_score"] < LOW_CONF_THRESHOLD

    # ---------- PERUBAHAN 1: Pola sarkasme diperluas konteks politik Indonesia ----------
    SARKAS_PATTERNS = [
        r"\bkatanya\b", r"\boh iya\b", r"\bsarkas", r"\bklasik\b", r"\boke sip\b",
        r"\byaudah lah\b", r"\bmaaf pak\b", r"\bpadahal\b", r"\bceritanya\b",
        r"\bterima kasih pak\b", r"\bkami (sangat )?bahagia\b", r"\bpov\b",
        r"terima kasih pak.*(gagal|rusak|kosong|susah|mahal|nunggak)",
        r"katanya (pemerintah|program|bansos|subsidi|gratis)",
        r"(meme|sindiran|sarkas)\b",
        r"\bjanji manis\b", r"\bomong kosong\b",
        r"\bmenteri bilang\b", r"\bpejabat bilang\b",
        r"\bkami mendengar aspirasi\b",
    ]
    sarkas_re = re.compile("|".join(SARKAS_PATTERNS), flags=re.I)

    df_medsos["sarkasme_flag"] = False
    mask = (df_medsos["sentiment"] == "Positif") & df_medsos["text_clean"].str.contains(sarkas_re)
    df_medsos.loc[mask, "sentiment"] = "Negatif"
    df_medsos.loc[mask, "sarkasme_flag"] = True

    # PERUBAHAN 2: Deteksi kritik keras terhadap pemerintah
    KRITIK_PATTERNS = [
        r"\bgagal\b", r"\bkorupsi\b", r"\bcurang\b", r"\bbohong\b", r"\bjanji\b",
        r"\bmahal\b", r"\brusak\b", r"\bparah\b", r"\bmenderita\b", r"\bsusah\b",
        r"\bkecewa\b", r"\bmengecewakan\b", r"\bprioritas apa\b", r"\bke mana\b",
        r"\bnggak (bisa|diterima|kebagian|adil)\b", r"\bpenindasan\b",
        r"\btikus berdasi\b", r"\bngeles\b", r"\bdirampas\b",
    ]
    kritik_re = re.compile("|".join(KRITIK_PATTERNS), flags=re.I)
    df_medsos["kritik_flag"] = df_medsos["text_clean"].str.contains(kritik_re)

    # PERUBAHAN 3: Deteksi apresiasi (positif eksplisit)
    APRESIASI_PATTERNS = [
        r"\bterima kasih\b.*(membantu|bermanfaat|bagus|cepat|lancar|keren|mantap)",
        r"\bsangat membantu\b", r"\bbermanfaat\b", r"\bmemudahkan\b",
        r"\blanjutkan\b", r"\bmantap\b", r"\bkeren\b", r"\bbagus banget\b",
        r"\bberjalan lancar\b", r"\bresponsif\b",
    ]
    apresiasi_re = re.compile("|".join(APRESIASI_PATTERNS), flags=re.I)
    df_medsos["apresiasi_flag"] = df_medsos["text_clean"].str.contains(apresiasi_re)

    print("✅ Analisis Sentimen Selesai!")
    print(f"   Dikoreksi sarkasme : {int(df_medsos['sarkasme_flag'].sum())} data")
    print(f"   Low confidence     : {int(df_medsos['low_confidence'].sum())} data")
    print(f"   Terdeteksi kritik  : {int(df_medsos['kritik_flag'].sum())} data")
    print(f"   Terdeteksi apresiasi: {int(df_medsos['apresiasi_flag'].sum())} data")

    print("\nDistribusi sentimen:")
    display(df_medsos["sentiment"].value_counts())
    if "platform" in df_medsos.columns:
        display(pd.crosstab(df_medsos["platform"], df_medsos["sentiment"]))
    if "topik_program" in df_medsos.columns:
        display(pd.crosstab(df_medsos["topik_program"], df_medsos["sentiment"]))

    display(df_medsos[[text_col, "topik_program", "sentiment", "sentiment_score",
                       "low_confidence", "sarkasme_flag", "meme_flag",
                       "kritik_flag", "apresiasi_flag"]].head(10))

In [ ]:
# Cell 5: Ringkasan Sentimen Kinerja Pemerintahan Indonesia
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- 1. PILIH DATA ----------
df = df_medsos.copy()
if "sumber" in df.columns and (df["sumber"] == "asli").sum() >= 30:
    df = df[df["sumber"] == "asli"]
    ket = "data asli"
    peringatan = ""
else:
    ket = "termasuk data dummy"
    peringatan = "⚠️ Hasil belum representatif: data asli < 30 / ada data dummy."

ORDER = ["Positif", "Netral", "Negatif"]
WARNA = {"Positif": "#4CAF50", "Netral": "#B0B0B0", "Negatif": "#E57373"}

total = len(df)
hitung = df["sentiment"].value_counts().reindex(ORDER, fill_value=0)
persen = hitung / total * 100

# ---------- 2. INDEKS SENTIMEN BERSIH (0-100) ----------
skor = {"Positif": 1, "Netral": 0, "Negatif": -1}
arr = df["sentiment"].map(skor).to_numpy()
indeks = arr.mean() * 50 + 50

rng = np.random.default_rng(42)
boot = [arr[rng.integers(0, total, total)].mean() * 50 + 50 for _ in range(2000)]
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

# PERUBAHAN 1: Ringkasan tambahan khusus kinerja pemerintahan
n_kritik = int(df["kritik_flag"].sum()) if "kritik_flag" in df.columns else 0
n_apresiasi = int(df["apresiasi_flag"].sum()) if "apresiasi_flag" in df.columns else 0
n_meme = int(df["meme_flag"].sum()) if "meme_flag" in df.columns else 0
n_sarkas = int(df["sarkasme_flag"].sum()) if "sarkasme_flag" in df.columns else 0

# ---------- 3. TEKS RINGKASAN ----------
print("=" * 55)
print(" RINGKASAN SENTIMEN KINERJA PEMERINTAHAN INDONESIA")
print("=" * 55)
print(f"Total data      : {total} ({ket})")
for s in ORDER:
    print(f"{s:<8}        : {hitung[s]:>4} data ({persen[s]:.1f}%)")
print("-" * 55)
print(f"Indeks Sentimen Bersih : {indeks:.1f}  (rentang 95%: {ci_low:.1f} - {ci_high:.1f})")
print(f"Kritik keras           : {n_kritik} data ({n_kritik/total*100:.1f}%)")
print(f"Apresiasi              : {n_apresiasi} data ({n_apresiasi/total*100:.1f}%)")
print(f"Konten meme/sindiran   : {n_meme} data ({n_meme/total*100:.1f}%)")
print(f"Dikoreksi sarkasme     : {n_sarkas} data")
if "low_confidence" in df.columns:
    print(f"Prediksi ragu-ragu     : {int(df['low_confidence'].sum())} data")
if peringatan:
    print(peringatan)
print("=" * 55)

# PERUBAHAN 2: Ringkasan per topik program
if "topik_program" in df.columns:
    print("\n📊 SENTIMEN PER TOPIK PROGRAM PEMERINTAHAN:")
    tbl_topik = pd.crosstab(df["topik_program"], df["sentiment"], normalize="index") * 100
    tbl_topik = tbl_topik.reindex(columns=ORDER, fill_value=0).round(1)
    tbl_topik["n"] = df["topik_program"].value_counts()
    display(tbl_topik.sort_values("n", ascending=False))

# PERUBAHAN 3: Perbandingan meme vs non-meme
if "meme_flag" in df.columns:
    print("\n📊 SENTIMEN: KONTEN MEME vs NON-MEME:")
    tbl_meme = pd.crosstab(df["meme_flag"], df["sentiment"], normalize="index") * 100
    tbl_meme = tbl_meme.reindex(columns=ORDER, fill_value=0).round(1)
    tbl_meme.index = ["Non-Meme", "Meme/Sindiran"]
    tbl_meme["n"] = df.groupby("meme_flag").size().values
    display(tbl_meme)

# ---------- 4. GRAFIK ----------
tampil_platform = "platform" in df.columns and df["platform"].nunique() > 1
n_plots = 2 + (1 if tampil_platform else 0) + (1 if "topik_program" in df.columns else 0)
fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4.5))
axes = np.atleast_1d(axes)

# Plot 1: Distribusi keseluruhan
ax = axes[0]
bars = ax.barh(ORDER[::-1], persen[ORDER[::-1]],
               color=[WARNA[s] for s in ORDER[::-1]])
for b, s in zip(bars, ORDER[::-1]):
    ax.text(b.get_width() + 1, b.get_y() + b.get_height() / 2,
            f"{persen[s]:.1f}% ({hitung[s]})", va="center", fontsize=10)
ax.set_xlim(0, max(persen.max() + 22, 50))
ax.set_xlabel("Persentase (%)")
ax.set_title("Distribusi Sentimen\nKinerja Pemerintahan", fontweight="bold")

# Plot 2: Per topik program (100% stacked)
plot_idx = 1
if "topik_program" in df.columns:
    ax2 = axes[plot_idx]
    tabel = pd.crosstab(df["topik_program"], df["sentiment"], normalize="index") * 100
    tabel = tabel.reindex(columns=ORDER, fill_value=0)
    tabel = tabel.loc[tabel.sum(axis=1).sort_values(ascending=False).index]
    kiri = np.zeros(len(tabel))
    for s in ORDER:
        ax2.barh(tabel.index, tabel[s], left=kiri, color=WARNA[s], label=s)
        kiri += tabel[s].to_numpy()
    ax2.set_xlim(0, 100)
    ax2.set_xlabel("Persentase (%)")
    ax2.set_title("Sentimen per Topik Program", fontweight="bold")
    ax2.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.18),
               frameon=False, fontsize=9)
    plot_idx += 1

# Plot 3: Per platform
if tampil_platform:
    ax3 = axes[plot_idx]
    tabel = pd.crosstab(df["platform"], df["sentiment"], normalize="index") * 100
    tabel = tabel.reindex(columns=ORDER, fill_value=0)
    kiri = np.zeros(len(tabel))
    for s in ORDER:
        ax3.barh(tabel.index, tabel[s], left=kiri, color=WARNA[s], label=s)
        for y, (v, l) in enumerate(zip(tabel[s], kiri)):
            if v >= 8:
                ax3.text(l + v / 2, y, f"{v:.0f}%", ha="center", va="center",
                         color="white", fontsize=9, fontweight="bold")
        kiri += tabel[s].to_numpy()
    ax3.set_xlim(0, 100)
    ax3.set_xlabel("Persentase (%)")
    ax3.set_title("Sentimen per Platform", fontweight="bold")
    ax3.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.18),
               frameon=False, fontsize=9)
    plot_idx += 1

# Plot 4: Meme vs Non-Meme
if "meme_flag" in df.columns:
    ax4 = axes[plot_idx]
    tbl_m = pd.crosstab(df["meme_flag"], df["sentiment"], normalize="index") * 100
    tbl_m = tbl_m.reindex(columns=ORDER, fill_value=0)
    tbl_m.index = ["Non-Meme", "Meme/Sindiran"]
    kiri = np.zeros(len(tbl_m))
    for s in ORDER:
        ax4.barh(tbl_m.index, tbl_m[s], left=kiri, color=WARNA[s], label=s)
        for y, (v, l) in enumerate(zip(tbl_m[s], kiri)):
            if v >= 8:
                ax4.text(l + v / 2, y, f"{v:.0f}%", ha="center", va="center",
                         color="white", fontsize=9, fontweight="bold")
        kiri += tbl_m[s].to_numpy()
    ax4.set_xlim(0, 100)
    ax4.set_xlabel("Persentase (%)")
    ax4.set_title("Meme vs Non-Meme", fontweight="bold")

for a in axes:
    a.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Sentimen Publik terhadap Kinerja Pemerintahan Indonesia "
             f"(n={total}, {ket})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
!pip install fpdf2


In [ ]:
from fpdf import FPDF
import matplotlib.pyplot as plt
import pandas as pd
import os

print("⏳ Sedang membuat laporan PDF...")

# 1. Simpan grafik dari Cell 5 menjadi file gambar sementara
# (Pastikan variabel 'fig' dari Cell 5 belum di-close/dihapus)
nama_file_grafik = "grafik_sentimen.png"
fig.savefig(nama_file_grafik, bbox_inches='tight', dpi=150)

# 2. Inisialisasi PDF (Format A4)
pdf = FPDF()
pdf.add_page()

# --- HALAMAN 1: RINGKASAN & GRAFIK ---
pdf.set_font("Arial", size=16, style='B')
pdf.cell(200, 10, txt="Laporan Analisis Sentimen Kinerja Pemerintahan", ln=True, align='C')
pdf.ln(5)

pdf.set_font("Arial", size=11)
pdf.cell(200, 8, txt=f"Total Data Dianalisis: {len(df_medsos)}", ln=True)
pdf.cell(200, 8, txt=f"Indeks Sentimen Bersih: {indeks:.1f} (Skala 0-100)", ln=True)
pdf.cell(200, 8, txt=f"Konten Meme/Sindiran: {int(df_medsos['meme_flag'].sum())} data", ln=True)
pdf.ln(10)

# Masukkan Gambar Grafik
# Mengatur lebar gambar agar pas di halaman A4 (max 190mm)
pdf.image(nama_file_grafik, x=10, y=50, w=190)

# --- HALAMAN 2: TABEL DATA ---
pdf.add_page()
pdf.set_font("Arial", size=14, style='B')
pdf.cell(200, 10, txt="Sampel Data Analisis (Top 30)", ln=True, align='C')
pdf.ln(5)

# Ambil 30 data teratas untuk dimasukkan ke tabel (agar PDF tidak terlalu tebal)
df_pdf = df_medsos[['platform', 'text', 'topik_program', 'sentiment']].head(30).copy()
# Potong teks agar tidak merusak layout tabel
df_pdf['text'] = df_pdf['text'].str.slice(0, 60) + "..."

# Konversi DataFrame ke HTML lalu render ke PDF
html_tabel = df_pdf.to_html(index=False, border=1, justify="left")
pdf.set_font("Arial", size=8) # Perkecil font agar tabel muat
pdf.write_html(html_tabel)

# 3. Simpan PDF
nama_file_pdf = "Laporan_Sentimen_Pemerintahan.pdf"
pdf.output(nama_file_pdf)
print(f"✅ Berhasil! File PDF tersimpan dengan nama: {nama_file_pdf}")

# 4. Hapus gambar sementara (opsional agar folder Colab bersih)
os.remove(nama_file_grafik)